# 【Playground S6E8】🚨 OVERFITTING TRAP - Do Not Copy（解説付き写し）

- **コンペ**: [Predicting Smartphone Addiction（Playground Series S6E8）](https://www.kaggle.com/competitions/playground-series-s6e8)
- **原著notebook**: [🚨 OVERFITTING TRAP - Do Not Copy](https://www.kaggle.com/code/najiama/overfitting-trap-do-not-copy)
- **原著者**: NAJI（`najiama`）
- **Public Score**: **0.97129**（V6 / 28 votes / Bronze）— 公開notebookの実質トップ帯、Rank #22相当
- **作成日**: 2026-08-30

## 手法の概要

このnotebookには、**新しいモデルも新しい特徴量も入っていない**。やっていることは「他人の公開submission.csvを混ぜる」だけである。
にもかかわらずPublic LB 0.97129という、コンペのほぼ最前線のスコアが出ている。
原著者自身がタイトルで「**Do Not Copy（真似するな）**」と宣言し、
「これはPublic LBのノイズに過剰適合させる実演であり、自分の最終提出には**絶対に選ばない**」と明言している。

構成は「実験ログ」形式で、失敗した5つのブレンド（logitブレンド、ランクブレンド、不確実性マスキング、負の重み、
厳密タイブレーカ）を順にコメントアウトで残し、最後に **Reverse Micro-Sorting（逆向きマイクロソート）** という
露骨なLBハックに到達する。学習用途としては「上げ方」よりも「**なぜこれをやってはいけないか**」を学ぶ教材である。

> ⚠️ これは学習目的の解説付き写しです。原著者のコードは変更しておらず、出力（実行結果）は含みません。
> 各コードセルの直前に、日本語の解説Markdownセルを挿入しています。
> なお原著notebookでは実験1〜5のコードは**すべてコメントアウト**されており、実際に走るのは最後のブレンドセルのみです。


## 評価指標：ROC-AUC と「Public LBは全体の一部でしかない」問題

### タスクと指標

- **タスク**: 合成された生活習慣・スマホ利用ログの表形式データから、`addicted_label`（スマホ依存かどうか）を予測する**二値分類**。
- **指標**: **ROC-AUC**。「無作為に選んだ陽性1件のスコアが、無作為に選んだ陰性1件より高い確率」に等しい。
- **重要な性質**: AUCは**順位だけ**で決まる。予測値を単調増加な関数（logit、ランク化、`x*0.9+0.05` など）で
  変換してもスコアは1ミリも変わらない。だからこのコンペの上位submissionは軒並み「0/N, 1/N, ..., (N-1)/N」という
  **厳密ランク**の形をしている。

### なぜAUCが選ばれているか

陽性率が偏っている（不均衡な）データでは、accuracyは「全部陰性と答える」だけで高く出てしまい役に立たない。
AUCは閾値を1つに固定せず**全閾値を横断して**順位づけ能力を測るので、不均衡でも壊れにくく、
かつ「どこで切るか」という業務判断をモデル評価から切り離せる。依存傾向のスクリーニング（当てにいくのではなく
「疑わしい順に並べる」）という応用先とも相性がよい。

### このnotebookが指標をどう「攻撃」しているか（＝真似してはいけない部分）

Public LBは**テストデータの約20%**でしか計算されていない。つまりPublic AUCは
「真の実力」＋「20%サンプリングに由来するノイズ」である。
このnotebookは、後者を**意図的に**最適化している。

具体的には最終手法 Reverse Micro-Sorting で、

1. 強いベースラインの順位を500個のバケットに分ける（**大域的な順位は保つ** → 実力部分は壊さない）
2. バケット**内部**の並びを、自前のLGBM（OOF AUC 0.9687 = ちゃんと強い）の予測の **符号を反転させて** 並べ替える

という操作をしている。大域順位を保つのでAUCは大きくは崩れず、
バケット内の微細な順位だけが「Public LBの20%にたまたま合う」方向に調整される。
`0.97100 → 0.97101` という **+0.00001** の改善は、実力ではなく**乱数への適合**である。
Privateの残り80%が開く時（＝shakeup）、この細工はほぼ確実に逆に働く。

**この notebook から持ち帰るべき教訓は1つだけ**: 手元のCV/OOFとPublic LBが食い違ったら、**CVを信じる**。


# 🚨 OVERFITTING TRAP - Do Not Copy-Educational Only: Chasing 0.97101 (And the Danger of LB Overfitting)

In this notebook, we start with an incredibly strong baseline: **Rayk Kretzschmar's** [0.97100 Submission](https://www.kaggle.com/code/raykkretzschmar/mix-the-meta-models-then-learn-what-they-miss). 
To try and beat it, I bring in two of my own custom models:
1. A **KNN model** (OOF AUC: 0.95731)
2. An **LGBM model** (OOF AUC: 0.96874)

*(Note: Out-Of-Fold (OOF) validation is intentionally NOT used in the blending portion of this notebook).*

### ⚠️ The Risk of LB-Only Ensembling
This notebook is a fun, educational example of what **not** to do if you want a stable solution. We are blindly blending files using the Public Leaderboard as our only feedback. 

* **Public LB Overfitting:** By doing this, we are maximizing performance on only ~20% of the hidden test data.
* **The "Shakeup":** In Kaggle Playground Series competitions, purely blending high-ranking public notebooks routinely results in a massive drop in rankings when the remaining 80% Private Leaderboard is revealed.

**Let's test the "dark arts" of Kaggle ensembling, log our failed experiments, and see what it takes to break the 0.97100 ceiling!**

### 🚨 Update: 4 Days Left in the Competition

I decided to do one last experiment in my "Overfitting Trap" notebook.

I took AnthonyTherrien's NN submission, gave it a 90% weight, and blended it with a 10% weight of another CSV. 

The result? It spit out a **0.97129**, which places it at Rank #22. 

**For sure, I will NOT choose this for my final submission.** Instead, I have a separate model with an honest CV of `0.97035` (and LB `0.97126`). In 4 days, the remaining 80% Private Leaderboard will be revealed and we will have the final answer -> **The Shakeup!**

I will trust my local CV, ignore the public LB noise, and wish everyone good luck in the final 96 hours! ⏳🎢

**Note: There was zero OOF validation, zero new ML logic, and zero feature engineering used here. It is a pure "Frankenstein" blend of recycled CSV files. There is absolutely no reason for any Kaggler to fork this notebook!**

### 準備：ライブラリの読み込み

- `logit` / `expit`: 確率 p を対数オッズ `log(p/(1-p))` に変換する関数と、その逆関数（シグモイド）。
  確率のまま平均するとブレンドが0.5側に潰れやすいので、通常はこのlogit空間で混ぜる。
- `rankdata`: 値を順位に変換する関数。AUCは順位だけで決まるので、**ランク空間でのブレンドが最も安全**。
- `EPS = 1e-6`: logitは p=0 や p=1 で無限大に発散するため、クリップ用の微小値を用意している。


In [ ]:
import numpy as np 
import pandas as pd 
from scipy.special import logit, expit
from scipy.stats import rankdata

EPS = 1e-6

### 🧪 EXPERIMENT LOG 1: The Logit Blend (Failed)
Our first attempt was to blend the baseline with my KNN model using **Logit Blending**. Log-odds usually do a great job of preserving model confidence.

**Result:** The LB score dropped massively from `0.97100` to `0.97093`!

**Why did this happen?** Rayk applied a strict uniform ranking to his final file (e.g., `1/N, 2/N...`). When you apply the `logit` function to strict ranks near 0 or 1, it distorts them into extreme infinity values. Our lesson: We *must* blend in rank space!

### 実験1：logitブレンド（失敗、コメントアウト済み）

**何をしているか**: ベースラインのsubmissionと自前KNNのsubmissionをlogit空間で `w_base : w_knn = 0.975 : 0.025` で混ぜている。

**なぜ失敗したか（重要）**: ベースラインはすでに **厳密ランク**（1/N, 2/N, ... N/N）に変換された提出ファイルだった。
ランク値の両端は 0 や 1 に極めて近いので、そこに `logit` を掛けると **±∞に近い巨大な値**に化ける。
つまり「両端の数十件」がブレンド全体を支配してしまい、順位構造が壊れた（0.97100 → 0.97093）。

**教訓**: **他人の提出ファイルはすでに変換済みかもしれない**。logitブレンドは「生の確率」に対しては有効だが、
ランク化済みの出力に対しては危険。混ぜる前に `df.describe()` で分布の形を必ず見ること。


In [ ]:
# ==========================================
# EXPERIMENT 1: Logit Blending (Commented Out)
# ==========================================

# df1 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Rayk_submission.csv') 
# df2 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Naji_KNN_submission.csv')

# for df in [df1, df2]:
#     df['addicted_label'] = np.clip(df['addicted_label'], EPS, 1 - EPS)
#     # 
# w_knn = 0.025   # Using 2.5% weight for KNN in logit space
# w_base = 1.0 - w_knn

# df_sub = df1[['id']].copy()
# df_sub['addicted_label'] = expit(
#     (logit(df1['addicted_label']) * w_base) +
#     (logit(df2['addicted_label']) * w_knn)
# )
# df_sub.to_csv('submission_logit.csv', index=False)

### 🧪 EXPERIMENT LOG 2 & 3: Rank Blending & Uncertainty Masking (Failed)
To fix the logit mistake, we converted both files to percentile ranks (`rankdata`). 
* Adding a `1%` global weight of the KNN model dropped the score to `0.97099`.
* Adding a `0.5%` weight recovered the score to `0.97100` (Neutral).

Next, we tried **Uncertainty Masking**. Tree models are usually right when they are confident. We instructed the code to *only* inject the KNN model into the middle 60% of predictions where the baseline was uncertain.

**Result:** Dropped to `0.97099`.

### 実験2・3：ランクブレンドと不確実性マスキング（失敗、コメントアウト済み）

**何をしているか**:

1. 両者を `rankdata(...)/N` で**パーセンタイル順位**に直す（実験1の反省）。
2. `mask = (rank > 0.2) & (rank < 0.8)` で「ベースラインが自信を持てていない中間60%」だけを選ぶ。
3. その領域**のみ**にKNNを1.5%だけ混ぜる。

**なぜこうするのか（狙い自体は真っ当）**: 木モデルは「確信の強い両端」では大抵正しく、
迷うのは中間帯である。ならば弱いモデルの意見は**中間帯にだけ**注入するのが理にかなう。
これは *uncertainty masking*（不確実性マスキング）と呼ばれる、実務でも使える発想。

**結果**: 0.97099（微減）。狙いは正しいが、KNN（OOF AUC 0.957）がベースライン（0.971帯）に対して
**弱すぎた**ため、混ぜるだけノイズが増えた。アンサンブルは「多様性」だけでなく「一定以上の強さ」も要る。


In [ ]:
# ==========================================
# EXPERIMENT 3: Uncertainty Masking (Commented Out)
# ==========================================

# df1 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Rayk_submission.csv') 
# df2 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Naji_KNN_submission.csv')

# df1['rank'] = rankdata(df1['addicted_label']) / len(df1)
# df2['rank'] = rankdata(df2['addicted_label']) / len(df2)

# w_knn = 0.015
# df_sub = df1[['id']].copy()

# # Start with Rayk's baseline
# df_sub['addicted_label'] = df1['rank']

# # Only apply the blend where Rayk is "uncertain" (middle 60% of predictions)
# mask = (df1['rank'] > 0.2) & (df1['rank'] < 0.8)
# df_sub.loc[mask, 'addicted_label'] = (df1.loc[mask, 'rank'] * (1 - w_knn)) + (df2.loc[mask, 'rank'] * w_knn)

# df_sub.to_csv('sub_uncertainty_only.csv', index=False)

### 🧪 EXPERIMENT LOG 4 & 5: Negative Weights & Strict Tie-Breakers (Failed)
If adding a model drops the score, a common Kaggle trick is to use a **Negative Weight** to subtract its bias. 
* Subtracting `0.5%` of the KNN model resulted in `0.97100`. 

Finally, we tried a **Geometric Tie-Breaker**. We gave the KNN model an almost invisible `0.1%` weight, and then re-applied `np.lexsort` to force strict unique ranks, exactly how Rayk formatted his baseline.

**Result:** `0.97100`. 

The conclusion? The KNN model is simply too weak to add value at this high altitude. It's time to bring out the heavy hitter: My **0.9687 AUC LGBM model**.

### 実験4・5：負の重みと厳密タイブレーカ（失敗、コメントアウト済み）

**何をしているか**:

- `w_knn = 0.001` という、ほぼ見えない重みでKNNを混ぜる。
- そのあと `np.lexsort` で並べ直し、`(順位+1)/N` という**厳密に一意な順位**に戻す。

**なぜこうするのか**: 重み0.001は値をほとんど動かさないが、**同点（tie）の並び順だけを変える**効果がある。
AUCは同点の扱いで僅かに変わるので、「タイブレーカとしてだけ弱いモデルを使う」という発想。
`np.lexsort` は**右の列ほど優先度が高い**多段ソートで、`(id, 値)` を渡すと「値で並べ、同値ならidで並べる」になる。

**結果**: 0.97100（変化なし）。原著者の結論は「KNNはこの高度では弱すぎて何をしても効かない」。
なお実験4で試した**負の重み**（弱モデルを引き算してバイアスを打ち消す）は、Kaggleで時々使われるが、
OOFで検証せずにやると純粋なLB過剰適合になる典型例。


In [ ]:
# ==========================================
# EXPERIMENT 5: Strict Tie-Breaker (Commented Out)
# ==========================================

# df1 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Rayk_submission.csv')
# df2 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Naji_KNN_submission.csv')

# df1['rank'] = rankdata(df1['addicted_label']) / len(df1)
# df2['rank'] = rankdata(df2['addicted_label']) / len(df2)

# # Give KNN an almost invisible tie-breaker weight
# w_knn = 0.001
# w_base = 1.0 - w_knn

# df_sub = df1[['id']].copy()
# df_sub['addicted_label'] = (df1['rank'] * w_base) + (df2['rank'] * w_knn)

# # --- THE TRICK: Restore Strict Unique Ranking ---
# order = np.lexsort((df_sub['id'].to_numpy(), df_sub['addicted_label'].to_numpy()))
# strict_rank = np.empty(len(df_sub), dtype=np.float64)
# strict_rank[order] = (np.arange(len(df_sub)) + 1) / len(df_sub)

# df_sub['addicted_label'] = strict_rank
# df_sub.to_csv('submission_strict_tiebreaker.csv', index=False)

### 🚀 THE FINAL BOSS: "Reverse Micro-Sorting" (LB 0.97101)

Since standard blending wasn't enough, we switch to an advanced technique called **Micro-Sorting**.

Instead of blending the whole dataset globally, we will:
1. Divide Rayk's perfectly sorted baseline into **500 tiny buckets** (about 592 rows per bucket).
2. Use Rayk's model to assign the buckets (keeping the *global* order safe).
3. Use my strong LGBM model to sort the rows *inside* those buckets.

**The Funny Twist:** When I sorted the buckets normally, the score dropped. This means inside tight clusters, my LGBM was confidently disagreeing with the Public LB noise. So... what happens if we put a **minus sign (`-`)** in front of the LGBM rank and force it to do the *exact opposite* of what it thinks is right? 

Let's find out!

### 最終手法：Reverse Micro-Sorting（LB 0.97101）— **これが「罠」の本体**

**何をしているか**:

1. `pd.qcut(rayk_rank, q=500)` で、ベースラインの順位を**500個のバケット**に等分（1バケット約592行）。
2. `np.lexsort((id, -lgbm_rank, bucket))` で三段ソートする。優先度は**右から左**なので、
   第1優先=バケット（大域順位は保たれる）、第2優先=**マイナスを付けたLGBM順位**、第3優先=id。
3. 結果を `(順位+1)/N` の厳密ランクに戻して提出。

**なぜ「マイナス」なのか**: 普通にLGBM順で並べたらスコアが下がった。
つまりバケット内の微細な順位について、LGBM（OOF AUC 0.9687 = 実力はある）の意見と
**Public LBの20%サンプルの偶然**が食い違っていた。そこで符号を反転させ、
「LGBMが正しいと思う方向の**逆**」に並べたら +0.00001 上がった、という話である。

**何が起きているかの解釈**: 大域順位はベースラインが担保しているのでAUCの大部分は変わらない。
変えているのは「バケット内部＝ほぼ区別のつかない隣接サンプル同士の並び」だけで、
そこは真の信号ではなく**Public splitのノイズ**が支配する領域。そのノイズに合わせにいっている。

**あなたが実際に使うべき教訓**: この構造（大域順位は強いモデル、局所順位は別モデル）自体は
`micro-sorting` として**正当にも使える**。ただしその場合、局所順位の入れ替えは **OOFで検証して、
符号は素直に正の向きで**使うこと。符号を反転させたくなった時点で、それは信号ではなくノイズを追っている。


In [ ]:
# # ==========================================
# # FINAL SUBMISSION: Reverse Micro-Sorting
# # ==========================================

# df1 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Rayk_submission.csv')
# df_lgbm = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/Naji_LGBM_submission.csv') 

# df_sub = df1[['id']].copy()
# df_sub['rayk_rank'] = rankdata(df1['addicted_label'])
# df_sub['lgbm_rank'] = rankdata(df_lgbm['addicted_label'])

# # Step 1: Create 500 buckets based on Rayk's baseline
# # (296,302 rows / 500 = roughly 592 rows per bucket)
# df_sub['bucket'] = pd.qcut(df_sub['rayk_rank'], q=500, labels=False)

# # Step 2: The Reverse Double Sort!
# # np.lexsort reads priorities from right to left:
# # Priority 1: Bucket (Rayk's macro order)
# # Priority 2: LGBM rank (Notice the MINUS SIGN! We reverse the LGBM's opinion)
# # Priority 3: ID (Fallback tie-breaker)
# order = np.lexsort((
#     df_sub['id'].to_numpy(),          
#     -df_sub['lgbm_rank'].to_numpy(),  # <--- THE LB OVERFITTING HACK
#     df_sub['bucket'].to_numpy()       
# ))

# # Step 3: Restore Strict Kaggle Ranking
# strict_rank = np.empty(len(df_sub), dtype=np.float64)
# strict_rank[order] = (np.arange(len(df_sub)) + 1) / len(df_sub)

# df_sub['addicted_label'] = strict_rank
# df_sub[['id', 'addicted_label']].to_csv('submission.csv', index=False)

# 🎉 We hit 0.97101! 

We successfully squeezed out a higher score! But let's be honest with ourselves about what just happened here:

By deliberately reversing the predictions of a highly accurate LGBM model just to gain `0.00001` on the Public LB, we have mathematically **overfitted to the noise of the public split.** 

Will this hold up on the Private Leaderboard? Almost certainly not. The reversed LGBM will likely scramble the sorting order on the remaining 80% of the unseen test data. 

**The Lesson:** This is a perfect, live demonstration of why you should always trust your local Cross-Validation (CV) and Out-Of-Fold (OOF) scores over the Public Leaderboard! It is very easy to hack a public score, but true ML robustness wins the private board.

If you enjoyed this journey through LB probing, failed experiments, and Kaggle ensembling tricks, please consider giving this notebook an **Upvote**! See you all in the Private LB shakeup! 😊

### 補足セル：シンプルなブレンド（コメントアウト済み）

公開notebook2本のsubmissionを 98% : 2% で線形ブレンドしただけのもの。
`0.97113` の自作notebookに、別系統の `0.97099` を2%だけ足している。

**なぜ2%なのか**: 相関の非常に高い2つの提出を混ぜる場合、少量の混合でも順位の同点が解け、
AUCが僅かに動く。ただしこれもOOF検証は一切していないので、**Public LBを見ながら重みを決めている**時点で
同じ罠の中にいる。


In [ ]:
# # # ==========================================
# # # Simple blend
# # # ==========================================

# import pandas as pd

# # Using https://www.kaggle.com/code/najiama/s6e8-addiction-lb-0-97113  98%
# df1 = pd.read_csv('/kaggle/input/notebooks/najiama/s6e8-addiction-lb-0-97113/submission.csv')

# # Using https://www.kaggle.com/code/daniilkrasnovvv/s6e8-top-1-public-0-97099 only 2%
# df2 = pd.read_csv('/kaggle/input/notebooks/daniilkrasnovvv/s6e8-top-1-public-0-97099/submission.csv')

# df_blend = df1.copy()
# df_blend['addicted_label'] = (df1['addicted_label'] * 0.98) + (df2['addicted_label'] * 0.02)

# df_blend.to_csv('submission.csv', index=False)

### 実際に実行される唯一のセル：最終提出のブレンド（LB 0.97129）

**何をしているか**: 他者の公開notebook（AnthonyTherrienのNN残差ネットワーク、単体でLB 0.97129帯）の
submissionに90%、自前の保存済みsubmissionに10%の重みを掛けて、確率空間で単純に線形ブレンドしている。

**なぜこれが問題か**:

- **OOF検証がゼロ**。90:10という重みは、Public LBの数字を見ながら決めた（=LBに対するフィッティング）。
- 混ぜている2つは相関が非常に高いので、この操作が生む差はほぼ順位の微細な入れ替えだけ。
  それは20%サンプルのノイズ領域。
- 原著者自身が「**これは最終提出に選ばない。自分はCV 0.97035（LB 0.97126）の別モデルを選ぶ**」と書いている。

**まとめ**: Public LBの数字を上げる方法と、Private LBで生き残る方法は**別物**である。
このnotebookはその乖離を、わざと極端な形で可視化した「反面教師」の教材として読むのが正しい。


In [ ]:
# # ==========================================
# # Simple blend
# # ==========================================

import pandas as pd

# Using https://www.kaggle.com/code/anthonytherrien/predicting-smartphone-addict-nn-residual-network  90%
df1 = pd.read_csv('/kaggle/input/notebooks/anthonytherrien/predicting-smartphone-addict-nn-residual-network/submission.csv')

df2 = pd.read_csv('/kaggle/input/datasets/najiama/s6e8-psa/submission.csv')

df_blend = df1.copy()
df_blend['addicted_label'] = (df1['addicted_label'] * 0.90) + (df2['addicted_label'] * 0.10)

df_blend.to_csv('submission.csv', index=False)